# 📐 Messaufgabe & Datenverarbeitungspipeline

Dieses Notebook führt Schritt für Schritt durch die Mess- und Auswertungsaufgabe. Ziel ist es, vorbereitete 3D-Messdaten (Punktewolken) zu laden, in Schnitte zu unterteilen und diese für eine Abrollsimulation vorzubereiten.

Import erforderlicher Module und Funktionen

In [1]:
# Notebook Setup
import glob
import time
import os
from pathlib import Path
import configparser

from Utils.Helping_Functions.config_init import config_setup
from Utils.Helping_Functions.get_files_to_process import get_files_to_process
from Utils.cut_layer_generator import cut_layer_generator

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Um aussagekräftige und robuste Ergebnisse zu erhalten werden bereits vermessene Zahnräder in die Pipeline geladen.
Unter "...\GearAnalysis-main\Data\Input\Measurements\Gear\Additional Measurements\" (abgespeichert unter gear_measurement_dir) liegen die dreidimensionalen Punktewolken als TXT Files ab.

In [2]:
# Laden der Konfiguration
config_setup()

# Laden der Konfiguration
config = configparser.ConfigParser()
config.read('config.ini')

# Zugriff auf die Pfade aus der Sektion 'dir'
gear_measurements_dir = Path(config.get('dir', 'gear_measurements_dir'))
evaluation_output_dir = Path(config.get('dir', 'evaluation_output_dir'))


## 1️⃣ Cut_Layer_Generator
- **Ziel:** Unterteilung der Punktwolke in gleichmäßig verteilte Schnittebenen (2D).
- **Input:** Originale Punktwolke
- **Output:** Separate 2D-Schnittebenen als NumPy-Arrays

In [5]:
# 📄 Liste der verfügbaren Messdateien
filepaths = list(gear_measurements_dir.glob("*.txt"))

# Wie viele Messdateien sind vorhanden?
total_files = len(filepaths)
print(f"Anzahl der Messdateien: {total_files}")

# Starte die Auswertung der ersten anzahl_messdateien = 5 Messdateien
anzahl_messdateien = 1
# for i, filepath in enumerate(filepaths[:anzahl_messdateien]):
for i, filepath in enumerate(filepaths):
    print(f"Starte die Auswertung der Datei {i + 1} von {anzahl_messdateien}: {filepath}")
    # Hier wird die Funktion aufgerufen, um die Schnitte zu generieren
    
    # Todo: Funktion noch manipulieren, um Aufgabe zu stellen
    cut_layer_generator(filepath)
    # print(f"Auswertung der Datei {i + 1} abgeschlossen.")
    print(f"\rCut Layer Generator: [{'#' * int(30 * (i + 1) / total_files):<30}] {i + 1}/{total_files} files processed", end='', flush=True)

Anzahl der Messdateien: 72
Starte die Auswertung der Datei 1 von 1: C:\Users\Administrator\Git Repos\GearAnalysis-main\Data\Input\Measurements\Gear\Additional Measurements\KW_10017.txt
Cut Layer Generator: [                              ] 1/72 files processedStarte die Auswertung der Datei 2 von 1: C:\Users\Administrator\Git Repos\GearAnalysis-main\Data\Input\Measurements\Gear\Additional Measurements\KW_10058.txt
Cut Layer Generator: [                              ] 2/72 files processedStarte die Auswertung der Datei 3 von 1: C:\Users\Administrator\Git Repos\GearAnalysis-main\Data\Input\Measurements\Gear\Additional Measurements\KW_6808.txt
Cut Layer Generator: [#                             ] 3/72 files processedStarte die Auswertung der Datei 4 von 1: C:\Users\Administrator\Git Repos\GearAnalysis-main\Data\Input\Measurements\Gear\Additional Measurements\KW_6842.txt
Cut Layer Generator: [#                             ] 4/72 files processedStarte die Auswertung der Datei 5 von 1: C:\Use

## 2️⃣ Cut_Layer_Converter
- **Ziel:** Konvertierung der Schnittebenen zur Weiterverarbeitung.
- **Schritte:**
  - Bereinigung & Interpolation
  - Formatierung für Reany-Kompatibilität
- **Output:** Strukturierte 2D-Daten, bereit zur Simulation

## 3️⃣ Reany-Abrollsimulation
- **Ziel:** Durchführung der Abrollsimulation anhand der vorbereiteten Schnitte.
- **Tool:** Reany (separater process)
- **Input:** Konvertierte Schnitte aus Schritt 2
- **Output:** Funktionsbasierte Analyseergebnisse analog zur Einflankenwälzprüfung


Nachdem die Schnitte generiert wurden, kann die restliche Pipeline über die .bat Datei "Schleife" auf dem Desktop getriggert werden.
Anschließend befinden sich die ausgewerteten Dateien als csv im Ordner: "C:\Users\Administrator\Desktop\Matlab_Code_Daniel\Reany_Output_CSV"



-------------------------------------------------------------------------------------------------------------------------------------

In [2]:
import subprocess
import os
from pathlib import Path
import time

# Pfade
input_folder = Path(r"C:\Users\Administrator\Desktop\Matlab_Code_Daniel\Input_Output\02_CLG_Output")
exe_folder = r"C:\Users\Administrator\Desktop\Matlab_Code_Daniel\CutLayerConverter"
exe_path = os.path.join(exe_folder, "CutLayerConverter.exe")

# Optional: Kontrolle, ob EXE vorhanden ist
if not os.path.exists(exe_path):
    raise FileNotFoundError(f"Die Datei {exe_path} wurde nicht gefunden.")

# Iteriere über alle .txt-Dateien im Input-Ordner
for txt_file in input_folder.glob("*.txt"):
    print(f"Verarbeite Datei: {txt_file.name}")
    print("Aktuelle Uhrzeit:", time.strftime("%H:%M:%S"))

    # Kommandos bauen
    cmd = [
        exe_path,
        "--filename", str(txt_file),
        "--inspection_plan", "KW_Z13",
        "--quit_after_protocol_creation"
    ]
    print(cmd)

    # Ausführen
    subprocess.run(cmd, cwd=exe_folder)
    
    # Warte 10 Sekunden
    time.sleep(10)

print("Alle Dateien wurden verarbeitet.")


Verarbeite Datei: KW_6808_cut_layers.txt
Aktuelle Uhrzeit: 15:53:29
['C:\\Users\\Administrator\\Desktop\\Matlab_Code_Daniel\\CutLayerConverter\\CutLayerConverter.exe', '--filename', 'C:\\Users\\Administrator\\Desktop\\Matlab_Code_Daniel\\Input_Output\\02_CLG_Output\\KW_6808_cut_layers.txt', '--inspection_plan', 'KW_Z13', '--quit_after_protocol_creation']


KeyboardInterrupt: 

In [1]:
import subprocess
import os

# Parameter
input_folder = r"C:\Users\Administrator\Desktop\Matlab_Code_Daniel\Input_Output\02_CLG_Output"
cutlayer_converter_path = r"C:\Users\Administrator\Desktop\Matlab_Code_Daniel\CutLayerConverter"
inspection_plan = "KW_Z13"

# Alle TXT-Dateien durchgehen
for file_name in os.listdir(input_folder):
    if file_name.endswith(".txt"):
        full_path = os.path.join(input_folder, file_name)

        print(f"Verarbeite: {file_name}")
        command = (
            f'cd /d "{cutlayer_converter_path}" && '
            f'CutLayerConverter.exe --filename "{full_path}" '
            f'--inspection_plan "{inspection_plan}" --quit_after_protocol_creation'
        )
        print(f"Command: {command}")
        subprocess.run(command, shell=True)
        subprocess.run("timeout /T 10", shell=True)


Verarbeite: KW_6808_cut_layers.txt
Command: cd /d "C:\Users\Administrator\Desktop\Matlab_Code_Daniel\CutLayerConverter" && CutLayerConverter.exe --filename "C:\Users\Administrator\Desktop\Matlab_Code_Daniel\Input_Output\02_CLG_Output\KW_6808_cut_layers.txt" --inspection_plan "KW_Z13" --quit_after_protocol_creation
Verarbeite: KW_7973_cut_layers.txt
Command: cd /d "C:\Users\Administrator\Desktop\Matlab_Code_Daniel\CutLayerConverter" && CutLayerConverter.exe --filename "C:\Users\Administrator\Desktop\Matlab_Code_Daniel\Input_Output\02_CLG_Output\KW_7973_cut_layers.txt" --inspection_plan "KW_Z13" --quit_after_protocol_creation


## 3️⃣ Reany-Abrollsimulation
- **Ziel:** Durchführung der Abrollsimulation anhand der vorbereiteten Schnitte.
- **Tool:** Reany (separater process)
- **Input:** Konvertierte Schnitte aus Schritt 2
- **Output:** Funktionsbasierte Analyseergebnisse analog zur Einflankenwälzprüfung

In [4]:
import subprocess

bat_file_path = r"C:\Users\Administrator\Git Repos\GearAnalysis-main\Schleife.bat"

# Starte die Batch-Datei und warte auf Beendigung
subprocess.run(['cmd', '/c', bat_file_path])

CompletedProcess(args=['cmd', '/c', 'C:\\Users\\Administrator\\Git Repos\\GearAnalysis-main\\Schleife.bat'], returncode=0)